# FlashAttention: Fast and Memory-Efficient Exact Attention with IO-Awareness

**Paper:** [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)  
**Authors:** Tri Dao, Daniel Y. Fu, Stefano Ermon, Atri Rudra, Christopher Ré  
**Venue:** NeurIPS 2022

This notebook implements a simplified version of the FlashAttention algorithm, demonstrating:
1. **Tiling** — processing Q, K, V in blocks instead of materializing the full N×N attention matrix
2. **Online softmax** — computing exact softmax incrementally across blocks using running statistics
3. **Memory & speed benchmarking** — comparing against naive attention at increasing sequence lengths

> **Note:** This implementation uses PyTorch ops (not fused CUDA kernels) for clarity. The memory advantage is real on any device. The speed advantage is most pronounced on GPUs where HBM bandwidth is the bottleneck.

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn.functional as F
import math
import time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

torch.manual_seed(42)
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device('cpu')
    print("Using CPU — memory advantage will be demonstrated; speed advantage may not be visible on CPU.")

print(f"PyTorch version: {torch.__version__}")

## 2. Naive Attention (Baseline)

Standard attention materializes the full N×N attention matrix in memory:
- S = QK^T ∈ R^{N×N}
- P = softmax(S) ∈ R^{N×N}
- O = PV ∈ R^{N×d}

This requires O(N²) memory per head, which becomes prohibitive for long sequences.

In [ ]:
def naive_attention(Q, K, V, causal=False):
    """Standard attention that materializes the full N×N attention matrix.
    
    Args:
        Q, K, V: (batch, heads, N, d)
        causal: if True, apply causal mask
    Returns:
        O: (batch, heads, N, d)
    """
    d = Q.shape[-1]
    scale = 1.0 / math.sqrt(d)
    S = torch.matmul(Q, K.transpose(-2, -1)) * scale  # (B, H, N, N)
    
    if causal:
        N = S.shape[-1]
        mask = torch.triu(torch.ones(N, N, device=S.device, dtype=torch.bool), diagonal=1)
        S = S.masked_fill(mask, float('-inf'))
    
    P = F.softmax(S, dim=-1)  # (B, H, N, N) — this is the N×N matrix we want to avoid
    O = torch.matmul(P, V)     # (B, H, N, d)
    return O

## 3. FlashAttention Forward (Tiled with Online Softmax)

The core algorithm:
1. **Tiling:** Split Q, K, V into blocks of size `block_size`
2. **Online softmax:** Track running max `m` and running sum `l` for each row
3. **Incremental output:** Accumulate the output `O` block by block, rescaling as the running max updates

The key mathematical insight is that softmax can be decomposed across blocks:
- When combining two blocks with maxima m₁ and m₂, the new max is max(m₁, m₂)
- The running sum and output must be rescaled by exp(old_max - new_max)

This avoids ever materializing the full N×N matrix in HBM.

In [ ]:
def flash_attention_forward(Q, K, V, block_size=64, causal=False):
    """FlashAttention forward pass with tiling and online softmax.
    
    Computes exact attention without materializing the full N×N attention matrix.
    
    Args:
        Q, K, V: (batch, heads, N, d)
        block_size: size of blocks for tiling (controls SRAM usage)
        causal: if True, apply causal mask
    Returns:
        O: (batch, heads, N, d)
    """
    B, H, N, d = Q.shape
    scale = 1.0 / math.sqrt(d)
    
    # Initialize output and online softmax statistics
    O = torch.zeros_like(Q)                     # (B, H, N, d)
    m = torch.full((B, H, N, 1), float('-inf'), device=Q.device, dtype=Q.dtype)  # running max
    l = torch.zeros(B, H, N, 1, device=Q.device, dtype=Q.dtype)                  # running sum
    
    # Outer loop: iterate over blocks of K, V
    for j_start in range(0, N, block_size):
        j_end = min(j_start + block_size, N)
        Kj = K[:, :, j_start:j_end, :]  # (B, H, block_size, d)
        Vj = V[:, :, j_start:j_end, :]  # (B, H, block_size, d)
        
        # Inner loop: iterate over blocks of Q
        for i_start in range(0, N, block_size):
            i_end = min(i_start + block_size, N)
            
            # Causal optimization: skip blocks entirely in the future
            if causal and j_start >= i_end:
                continue
            
            Qi = Q[:, :, i_start:i_end, :]  # (B, H, block_size, d)
            
            # Compute partial attention scores for this block
            Sij = torch.matmul(Qi, Kj.transpose(-2, -1)) * scale  # (B, H, block_size, block_size)
            
            if causal:
                # Apply causal mask within the block
                block_N = i_end - i_start
                block_M = j_end - j_start
                # Create causal mask for this sub-block
                rows = torch.arange(block_N, device=Q.device).unsqueeze(1) + i_start
                cols = torch.arange(block_M, device=Q.device).unsqueeze(0) + j_start
                mask = cols > rows  # mask out future positions
                Sij = Sij.masked_fill(mask.unsqueeze(0).unsqueeze(0), float('-inf'))
            
            # Online softmax: compute block-local max and sum
            mij = Sij.max(dim=-1, keepdim=True).values           # (B, H, block_size, 1)
            # Handle all-masked rows (mij = -inf) to avoid NaN
            mij_safe = torch.where(torch.isinf(mij), torch.zeros_like(mij), mij)
            Pij = torch.exp(Sij - mij_safe)                       # (B, H, block_size, block_size)
            # Zero out rows that were entirely masked
            all_masked = torch.isinf(mij).squeeze(-1)  # (B, H, block_size)
            Pij = Pij.masked_fill(all_masked.unsqueeze(-1), 0.0)
            lij = Pij.sum(dim=-1, keepdim=True)                   # (B, H, block_size, 1)
            
            # Get current running stats for this Q block
            mi_old = m[:, :, i_start:i_end, :]
            li_old = l[:, :, i_start:i_end, :]
            Oi_old = O[:, :, i_start:i_end, :]
            
            # Update running max
            mi_new = torch.max(mi_old, mij)
            
            # Rescale old statistics and accumulate new
            alpha = torch.exp(mi_old - mi_new)   # rescale factor for old
            beta = torch.exp(mij - mi_new)        # rescale factor for new
            
            li_new = alpha * li_old + beta * lij
            Oi_new = (alpha * li_old * Oi_old + beta * torch.matmul(Pij, Vj)) / li_new
            
            # Write back
            O[:, :, i_start:i_end, :] = Oi_new
            m[:, :, i_start:i_end, :] = mi_new
            l[:, :, i_start:i_end, :] = li_new
    
    return O

## 4. Correctness Verification

We verify that FlashAttention produces the same output as:
1. Naive attention (our baseline)
2. PyTorch's built-in `scaled_dot_product_attention`

The outputs should match within floating-point tolerance.

In [ ]:
def verify_correctness():
    """Verify FlashAttention matches naive attention and PyTorch SDPA."""
    B, H, N, d = 1, 2, 128, 64
    Q = torch.randn(B, H, N, d, device=device)
    K = torch.randn(B, H, N, d, device=device)
    V = torch.randn(B, H, N, d, device=device)
    
    # Naive attention
    O_naive = naive_attention(Q, K, V)
    
    # FlashAttention
    O_flash = flash_attention_forward(Q, K, V, block_size=32)
    
    # PyTorch SDPA (ground truth from PyTorch)
    O_sdpa = F.scaled_dot_product_attention(Q, K, V)
    
    # Compare
    diff_naive_flash = (O_naive - O_flash).abs().max().item()
    diff_naive_sdpa = (O_naive - O_sdpa).abs().max().item()
    diff_flash_sdpa = (O_flash - O_sdpa).abs().max().item()
    
    print(f"Max |naive - flash|:  {diff_naive_flash:.2e}")
    print(f"Max |naive - sdpa|:   {diff_naive_sdpa:.2e}")
    print(f"Max |flash - sdpa|:   {diff_flash_sdpa:.2e}")
    
    assert diff_naive_flash < 1e-4, f"FlashAttention differs from naive by {diff_naive_flash}"
    print("\n✓ Correctness verified: FlashAttention matches naive attention within tolerance!")

verify_correctness()

### 4.1 Correctness with Causal Masking

In [ ]:
def verify_causal_correctness():
    """Verify FlashAttention with causal mask matches naive with causal mask."""
    B, H, N, d = 1, 2, 64, 32
    Q = torch.randn(B, H, N, d, device=device)
    K = torch.randn(B, H, N, d, device=device)
    V = torch.randn(B, H, N, d, device=device)
    
    O_naive_causal = naive_attention(Q, K, V, causal=True)
    O_flash_causal = flash_attention_forward(Q, K, V, block_size=16, causal=True)
    
    diff = (O_naive_causal - O_flash_causal).abs().max().item()
    print(f"Max |naive_causal - flash_causal|: {diff:.2e}")
    assert diff < 1e-4, f"Causal FlashAttention differs by {diff}"
    print("✓ Causal masking verified!")

verify_causal_correctness()

## 5. Memory Benchmark

We compare peak memory usage between naive and FlashAttention at increasing sequence lengths. 

- **Naive attention:** O(N²) memory per head (the S and P matrices)
- **FlashAttention:** O(Nd) memory per head (only Q, K, V, O are stored; blocks are small)

On GPU, we use `torch.cuda.max_memory_allocated()`. On CPU, we estimate memory by tracking tensor sizes.

In [ ]:
def estimate_attention_memory(N, d, H=1, B=1):
    """Estimate memory for naive attention (the N×N matrices are the key cost)."""
    # Q, K, V: B*H*N*d each
    qkv_mem = 3 * B * H * N * d * 4  # float32 = 4 bytes
    # S and P: B*H*N*N each (these are the expensive ones)
    sp_mem = 2 * B * H * N * N * 4
    # O: B*H*N*d
    o_mem = B * H * N * d * 4
    total_naive = qkv_mem + sp_mem + o_mem
    
    # Flash: only Q, K, V, O + small block buffers
    total_flash = qkv_mem + o_mem  # no S, P matrices
    
    return total_naive, total_flash

def measure_gpu_memory(fn, *args, **kwargs):
    """Measure peak GPU memory for a function call."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    _ = fn(*args, **kwargs)
    return torch.cuda.max_memory_allocated()

def run_memory_benchmark(seq_lengths, d=64, H=4, B=1):
    """Benchmark memory usage at different sequence lengths."""
    results = {'N': [], 'naive_actual': [], 'flash_actual': [], 'naive_est': [], 'flash_est': []}
    
    for N in seq_lengths:
        print(f"  Testing N={N}...")
        Q = torch.randn(B, H, N, d, device=device)
        K = torch.randn(B, H, N, d, device=device)
        V = torch.randn(B, H, N, d, device=device)
        
        # Estimated memory
        naive_est, flash_est = estimate_attention_memory(N, d, H, B)
        results['naive_est'].append(naive_est / 1e6)  # MB
        results['flash_est'].append(flash_est / 1e6)
        
        if device.type == 'cuda':
            # Actual GPU memory
            naive_mem = measure_gpu_memory(naive_attention, Q, K, V)
            flash_mem = measure_gpu_memory(flash_attention_forward, Q, K, V, block_size=64)
            results['naive_actual'].append(naive_mem / 1e6)
            results['flash_actual'].append(flash_mem / 1e6)
        else:
            # On CPU, use estimated memory (can't easily measure actual RSS)
            results['naive_actual'].append(naive_est / 1e6)
            results['flash_actual'].append(flash_est / 1e6)
        
        results['N'].append(N)
        
        # Clean up
        del Q, K, V
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    return results

seq_lengths = [256, 512, 1024, 2048, 4096, 8192]
print("Running memory benchmark...")
mem_results = run_memory_benchmark(seq_lengths, d=64, H=4, B=1)
print("Done!")

In [ ]:
# Plot memory benchmark
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Memory plot
ax1.plot(mem_results['N'], mem_results['naive_actual'], 'ro-', label='Naive (actual)', markersize=8)
ax1.plot(mem_results['N'], mem_results['flash_actual'], 'bs-', label='Flash (actual)', markersize=8)
ax1.plot(mem_results['N'], mem_results['naive_est'], 'r--', label='Naive (estimated)', alpha=0.5)
ax1.plot(mem_results['N'], mem_results['flash_est'], 'b--', label='Flash (estimated)', alpha=0.5)
ax1.set_xlabel('Sequence Length (N)')
ax1.set_ylabel('Memory (MB)')
ax1.set_title('Memory Usage: Naive vs FlashAttention')
ax1.legend()
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3)

# Memory ratio
ratios = [n / f for n, f in zip(mem_results['naive_actual'], mem_results['flash_actual'])]
ax2.bar(range(len(ratios)), ratios, color=['#e74c3c' if r > 1 else '#3498db' for r in ratios])
ax2.set_xticks(range(len(ratios)))
ax2.set_xticklabels([str(n) for n in mem_results['N']])
ax2.set_xlabel('Sequence Length (N)')
ax2.set_ylabel('Naive / Flash Memory Ratio')
ax2.set_title('Memory Savings Ratio (higher = more savings)')
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('memory_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()
print("Memory benchmark plot saved!")

# Print table
print("\nMemory Benchmark Results:")
print(f"{'N':>6} | {'Naive (MB)':>12} | {'Flash (MB)':>12} | {'Ratio':>8} | {'Savings':>10}")
print("-" * 60)
for i, N in enumerate(mem_results['N']):
    n_mem = mem_results['naive_actual'][i]
    f_mem = mem_results['flash_actual'][i]
    ratio = n_mem / f_mem if f_mem > 0 else float('inf')
    savings = (1 - f_mem / n_mem) * 100 if n_mem > 0 else 0
    print(f"{N:>6} | {n_mem:>12.1f} | {f_mem:>12.1f} | {ratio:>8.2f} | {savings:>9.1f}%")

## 6. Speed Benchmark

We compare wall-clock time between naive and FlashAttention at increasing sequence lengths.

> **Important:** On CPU, FlashAttention may be *slower* than naive because:
> - There's no HBM/SRAM distinction on CPU (all memory is equivalent)
> - The tiled version does extra work (block management, rescaling)
> - The benefit of reduced memory access doesn't apply
>
> On GPU, FlashAttention is faster because HBM bandwidth is the bottleneck.
> The notebook still demonstrates the algorithmic principle regardless of device.

In [ ]:
def benchmark_fn(fn, *args, warmup=3, repeats=5, **kwargs):
    """Benchmark a function with warmup runs."""
    # Warmup
    for _ in range(warmup):
        _ = fn(*args, **kwargs)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    times = []
    for _ in range(repeats):
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        _ = fn(*args, **kwargs)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        times.append(end - start)
    
    return sum(times) / len(times)  # mean

def run_speed_benchmark(seq_lengths, d=64, H=4, B=1):
    """Benchmark speed at different sequence lengths."""
    results = {'N': [], 'naive_time': [], 'flash_time': [], 'speedup': []}
    
    for N in seq_lengths:
        print(f"  Testing N={N}...")
        Q = torch.randn(B, H, N, d, device=device)
        K = torch.randn(B, H, N, d, device=device)
        V = torch.randn(B, H, N, d, device=device)
        
        # Naive
        t_naive = benchmark_fn(naive_attention, Q, K, V, warmup=2, repeats=3)
        
        # Flash
        t_flash = benchmark_fn(flash_attention_forward, Q, K, V, block_size=64, warmup=2, repeats=3)
        
        speedup = t_naive / t_flash if t_flash > 0 else 0
        
        results['N'].append(N)
        results['naive_time'].append(t_naive)
        results['flash_time'].append(t_flash)
        results['speedup'].append(speedup)
        
        print(f"    Naive: {t_naive*1000:.1f}ms, Flash: {t_flash*1000:.1f}ms, Speedup: {speedup:.2f}x")
        
        del Q, K, V
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    
    return results

# Use smaller sequence lengths for speed benchmark on CPU
if device.type == 'cuda':
    speed_seq_lengths = [256, 512, 1024, 2048, 4096]
else:
    speed_seq_lengths = [256, 512, 1024, 2048]

print("Running speed benchmark...")
speed_results = run_speed_benchmark(speed_seq_lengths, d=64, H=4, B=1)
print("Done!")

In [ ]:
# Plot speed benchmark
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Time plot
ax1.plot(speed_results['N'], [t*1000 for t in speed_results['naive_time']], 'ro-', label='Naive', markersize=8)
ax1.plot(speed_results['N'], [t*1000 for t in speed_results['flash_time']], 'bs-', label='Flash', markersize=8)
ax1.set_xlabel('Sequence Length (N)')
ax1.set_ylabel('Time (ms)')
ax1.set_title('Wall-Clock Time: Naive vs FlashAttention')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Speedup plot
colors = ['#2ecc71' if s > 1 else '#e74c3c' for s in speed_results['speedup']]
ax2.bar(range(len(speed_results['speedup'])), speed_results['speedup'], color=colors)
ax2.set_xticks(range(len(speed_results['speedup'])))
ax2.set_xticklabels([str(n) for n in speed_results['N']])
ax2.set_xlabel('Sequence Length (N)')
ax2.set_ylabel('Speedup (Naive / Flash time)')
ax2.set_title('Speedup Ratio (>1 = Flash is faster)')
ax2.axhline(y=1, color='gray', linestyle='--', alpha=0.5)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('speed_benchmark.png', dpi=100, bbox_inches='tight')
plt.show()
print("Speed benchmark plot saved!")

# Print table
print("\nSpeed Benchmark Results:")
print(f"{'N':>6} | {'Naive (ms)':>12} | {'Flash (ms)':>12} | {'Speedup':>8}")
print("-" * 48)
for i, N in enumerate(speed_results['N']):
    print(f"{N:>6} | {speed_results['naive_time'][i]*1000:>12.1f} | {speed_results['flash_time'][i]*1000:>12.1f} | {speed_results['speedup'][i]:>7.2f}x")

## 7. Analysis & Discussion

### Key Observations

1. **Memory:** FlashAttention uses dramatically less memory than naive attention, especially at long sequence lengths. The memory savings grow with N because naive attention stores O(N²) while FlashAttention stores O(Nd).

2. **Speed on GPU vs CPU:**
   - On **GPU**: FlashAttention is faster because it reduces HBM reads/writes, which are the bottleneck. The algorithm trades extra FLOPs (from recomputation) for fewer memory accesses — a net win when memory-bound.
   - On **CPU**: FlashAttention may be slower because there's no HBM/SRAM distinction. The extra computation (block management, rescaling) adds overhead without the corresponding memory-access savings. The memory advantage is still real, but the speed advantage may not appear.

3. **The IO-Awareness Principle:** The fundamental insight is that on modern hardware, **memory access patterns matter more than FLOP count**. An algorithm that does more arithmetic but fewer memory accesses can be faster — this is the opposite of traditional algorithm analysis which focuses on computational complexity.

### IO Complexity

| Metric | Naive Attention | FlashAttention |
|--------|----------------|----------------|
| HBM accesses | Ω(Nd + N²) | O(N²d²/M) |
| Memory | O(N²) | O(Nd) |
| FLOPs | O(N²d) | O(N²d) (same) |

Where M is SRAM size and d is head dimension. For typical values (d=64, M=100KB), FlashAttention requires ~9× fewer HBM accesses.

In [ ]:
# Summary comparison table
print("=" * 70)
print("FLASHATTENTION vs NAIVE ATTENTION — SUMMARY")
print("=" * 70)
print()
print(f"{'Metric':<25} | {'Naive':<20} | {'FlashAttention':<20}")
print("-" * 70)
print(f"{'Memory complexity':<25} | {'O(N²)':<20} | {'O(Nd)':<20}")
print(f"{'HBM accesses':<25} | {'Ω(Nd + N²)':<20} | {'O(N²d²/M)':<20}")
print(f"{'FLOPs':<25} | {'O(N²d)':<20} | {'O(N²d) — same':<20}")
print(f"{'Exact result?':<25} | {'Yes':<20} | {'Yes':<20}")
print(f"{'Materializes N×N?':<25} | {'Yes':<20} | {'No':<20}")
print(f"{'Kernel fusion':<25} | {'No':<20} | {'Yes (in CUDA impl)':<20}")
print()
print("At N=4096, d=64, H=4:")
naive_mem_4096 = 4 * 4096 * 4096 * 4 * 2 / 1e6  # S + P matrices, float32
flash_mem_4096 = 0  # no S, P
print(f"  Naive S+P matrices: {naive_mem_4096:.1f} MB (per batch)")
print(f"  Flash S+P matrices: 0 MB (never materialized)")
print(f"  Memory saved: {naive_mem_4096:.1f} MB")
print()
print("✓ FlashAttention computes the EXACT same result as standard attention,")
print("  just with dramatically fewer memory accesses.")

## 8. Bonus: Causal Masking & Block-Sparse Extension

The paper extends FlashAttention to block-sparse attention, where entire blocks of the attention matrix are skipped (set to zero). This is particularly useful for causal masking, where blocks above the diagonal can be entirely skipped.

In the causal case, approximately half the blocks are zero, yielding ~2× additional speedup. For general sparsity patterns (e.g., local window attention), even more blocks can be skipped.

In [ ]:
def count_causal_blocks(N, block_size):
    """Count how many blocks are needed for causal vs full attention."""
    num_blocks = (N + block_size - 1) // block_size
    total_blocks = num_blocks * num_blocks
    # Causal: only lower triangular blocks (including diagonal)
    causal_blocks = num_blocks * (num_blocks + 1) // 2
    skipped = total_blocks - causal_blocks
    return total_blocks, causal_blocks, skipped

print("Block-Sparse Causal Attention — Block Savings:")
print(f"{'N':>6} | {'Block Size':>10} | {'Total Blocks':>12} | {'Causal Blocks':>14} | {'Skipped':>8} | {'Savings':>8}")
print("-" * 72)
for N in [512, 1024, 2048, 4096, 8192]:
    bs = 64
    total, causal, skipped = count_causal_blocks(N, bs)
    savings = skipped / total * 100
    print(f"{N:>6} | {bs:>10} | {total:>12} | {causal:>14} | {skipped:>8} | {savings:>7.1f}%")

print()
print("With block-sparse causal masking, approximately half the blocks are skipped,")
print("giving roughly 2× additional speedup on top of FlashAttention's base speedup.")

## 9. Conclusion

FlashAttention demonstrates that **IO-awareness** — carefully accounting for memory hierarchy — is more important than FLOP reduction for attention on modern hardware. The key innovations are:

1. **Tiling** to avoid materializing the N×N attention matrix
2. **Online softmax** to compute exact softmax incrementally across blocks  
3. **Recomputation** in the backward pass to trade extra FLOPs for fewer memory accesses

This notebook implemented the forward pass with tiling and online softmax, verified correctness against standard attention, and benchmarked memory and speed. The memory savings are dramatic and real on any device; the speed savings are most pronounced on GPUs where HBM bandwidth is the bottleneck.

### Real-World Impact

FlashAttention is now integrated into PyTorch 2.0+ as `scaled_dot_product_attention`, HuggingFace Transformers, and virtually all modern LLM training pipelines. It enabled the training of models with context lengths of 16K-64K, which would have been infeasible with standard attention.

**Paper:** [arXiv:2205.14135](https://arxiv.org/abs/2205.14135)  
**Code:** [github.com/Dao-AILab/flash-attention](https://github.com/Dao-AILab/flash-attention)  
**Kaggle:** [![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/nadymsazad/flashattention-fast-memory-efficient-exact-attention)